### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="heart_disease_cleveland",
    dataset_year="1989",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C52P4X",
    download_description="""
Get the UCI data.

wget https://archive.ics.uci.edu/static/public/45/heart+disease.zip && unzip heart+disease.zip processed.cleveland.data && rm heart+disease.zip && mkdir -p local-data-warehouse/heart_disease_cleveland && mv processed.cleveland.data local-data-warehouse/heart_disease_cleveland/
""",
    # References
    academic_reference_bibtex="""@article{detrano1989international,
  title={International application of a new probability algorithm for the diagnosis of coronary artery disease},
  author={Detrano, Robert and Janosi, Andras and Steinbrunn, Walter and Pfisterer, Matthias and Schmid, Johann-Jakob and Sandhu, Sarbjit and Guppy, Kern H and Lee, Stella and Froelicher, Victor},
  journal={The American journal of cardiology},
  volume={64},
  number={5},
  pages={304--310},
  year={1989},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="detrano1989international",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the processed version and the subset of 14 attributes used in the study and clinical practice.

- We encode missing values as np.nan instead of "?".
- We make the target binary (0=no heart disease, 1=heart disease). This follows the original study in attempting to distinguish presence (values 1,2,3,4) from absence (value 0).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="heart_disease_diagnosis",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="heart_disease_diagnosis",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "num"
]

df = pd.read_csv(dataset_mold.path / "processed.cleveland.data", header=None, names=columns)
print("Loaded data shape:", df.shape)

df = df.replace("?", np.nan)
df["ca"] = df["ca"].astype(float)
# Make target
df["heart_disease_diagnosis"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

as_cat_type = ["thal", "slope", "exang", "restecg", "fbs", "cp", "sex", "heart_disease_diagnosis"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (303, 14)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 303
Columns: 14
Use sampling: False (sample size: 303)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['chol', 'thalach', 'trestbps', 'age', 'oldpeak', 'cp', 'ca', 'thal', 'slope', 'restecg']
Rows remaining as candidates after top-10 filter: 0 (of 303)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,heart_disease_diagnosis
0,53.0,1.0,3.0,130.0,246.0,1.0,2.0,173.0,0.0,0.0,1.0,3.0,3.0,0
1,54.0,1.0,4.0,110.0,206.0,0.0,2.0,108.0,1.0,0.0,2.0,1.0,3.0,1
2,56.0,1.0,4.0,125.0,249.0,1.0,2.0,144.0,1.0,1.2,2.0,1.0,3.0,1
3,58.0,1.0,4.0,100.0,234.0,0.0,0.0,156.0,0.0,0.1,1.0,1.0,7.0,1
4,51.0,0.0,4.0,130.0,305.0,0.0,0.0,142.0,1.0,1.2,2.0,0.0,7.0,1


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,thal,category,2.0,0.66,3.0,"3.0, 7.0, 6.0"
1,sex,category,0.0,0.00,2.0,"1.0, 0.0"
2,cp,category,0.0,0.00,4.0,"4.0, 3.0, 2.0, 1.0"
3,fbs,category,0.0,0.00,2.0,"0.0, 1.0"
4,restecg,category,0.0,0.00,3.0,"0.0, 2.0, 1.0"
5,exang,category,0.0,0.00,2.0,"0.0, 1.0"
6,slope,category,0.0,0.00,3.0,"1.0, 2.0, 3.0"
7,heart_disease_diagnosis,category,0.0,0.00,2.0,"0, 1"
8,ca,float64,4.0,1.32,4.0,"0.0, 1.0, 2.0, 3.0"
9,age,float64,0.0,0.00,41.0,"58.0, 57.0, 54.0, 59.0, 52.0, 51.0, 60.0, 62.0, 56.0, 44.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,303.0,54.438944,9.038662,29.0,77.0
trestbps,303.0,131.689769,17.599748,94.0,200.0
chol,303.0,246.693069,51.776918,126.0,564.0
thalach,303.0,149.607261,22.875003,71.0,202.0
oldpeak,303.0,1.039604,1.161075,0.0,6.2
ca,299.0,0.672241,0.937438,0.0,3.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                  rank                    
cp                      1      4.0    144  47.52
                        2      3.0     86  28.38
                        3      2.0     50  16.50
                        4      1.0     23   7.59
exang                   1      0.0    204  67.33
                        2      1.0     99  32.67
fbs                     1      0.0    258  85.15
                        2      1.0     45  14.85
heart_disease_diagnosis 1        0    164  54.13
                        2        1    139  45.87
restecg                 1      0.0    151  49.83
                        2      2.0    148  48.84
                        3      1.0      4   1.32
sex                     1      1.0    206  67.99
                        2      0.0     97  32.01
slope                   1      1.0    142  46.86
                        2      2.0    140  46.20
                        3      3.0     21   6.93
thal                    1      3.0    166  54.79
                        2      7.0    117  38.61
                        3      6.0     18   5.94
                        4     <NA>      2   0.66

In [8]:
# Target Distribution
target_df

,count,pct
heart_disease_diagnosis,,
0,164,54.13
1,139,45.87


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7513-909c-707d-a9da-9852f346a015
f61b1f424026107e2aa24e6210adb50d778c3bb1bc879f0a61958cc904b00a38
